# Phase 2.5 (split B): GBT — Flight delay prediction

**Group 01-02 | DS261 Spring 2026**

This notebook is the **memory/workflow split** counterpart to [phase2.5_modeling_StandardScaler_LR_RF_undersample.ipynb](phase2.5_modeling_StandardScaler_LR_RF_undersample.ipynb). It loads **only** Notebook **B** ML parquets (`phase2_df_rawasm_undersample_*_ml.parquet`) from [Phase2.4_Pipeline_AssemblerOnly_ML_undersample.ipynb](Phase2.4_Pipeline_AssemblerOnly_ML_undersample.ipynb).

**Use the all-in-one** [phase2.5_modeling_final.ipynb](phase2.5_modeling_final.ipynb) if you want one run with a combined summary table and enough cluster memory.

**Pipeline context:** F-β (β=2), GBT on Phase 3.3 train parquet (single undersample; no resample here), grid search, threshold tuning, time-series CV, blind test, feature importance.

**Why split?** Caching train/val/test for **both** A and B paths in one session doubles parquet cache pressure. This notebook caches **three** frames (B only).

See [Phase2.4_Pipeline_ML_split.md](Phase2.4_Pipeline_ML_split.md) for Phase 2.4 → 2.5 path mapping. **Notebook A** (LR/RF path) saves its fitted `PipelineModel` to `{BASE_GROUP}/phase3_pipeline_model_standard_undersample` when `WRITE_PIPELINE_MODEL=True` (this notebook does not load that path).



In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import time

from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.functions import vector_to_array

# ── Configuration ──────────────────────────────────────────────────────────────
BASE_GROUP = "dbfs:/student-groups/Group_01_02"
TARGET     = "DEP_DEL15"
BETA       = 2.0   # F-β weight: recall valued 2× over precision
VERIFY_ROW_COUNTS = False  # True: extra count() diagnostics on load (memory heavy)

# Notebook B (assembler-only) ML parquets only
ML_PREFIX_B = "phase2_df_rawasm_undersample"
TRAIN_ML_PATH_B = f"{BASE_GROUP}/{ML_PREFIX_B}_train_ml.parquet"
VAL_ML_PATH_B   = f"{BASE_GROUP}/{ML_PREFIX_B}_val_ml.parquet"
TEST_ML_PATH_B  = f"{BASE_GROUP}/{ML_PREFIX_B}_test_ml.parquet"

print("Libraries loaded (GBT split notebook).")
print(f"F-β metric: β = {BETA}  (recall weighted {int(BETA)}× over precision)")



Libraries loaded (GBT split notebook).
F-β metric: β = 2.0  (recall weighted 2× over precision)


## 1. Data Loading & Verification

Load ML-ready parquets from **Phase 2.4 Notebook B** ([Phase2.4_Pipeline_AssemblerOnly_ML_undersample.ipynb](Phase2.4_Pipeline_AssemblerOnly_ML_undersample.ipynb)): `features` assembled from parquet numerics (**no** train-fitted `StandardScaler` on non-`*_scl`; `*_scl` unchanged vs Phase 3.3) and `DEP_DEL15`. Any `class_weight` column is ignored by GBT.



In [0]:
# Alias B paths to df_train_ml / df_val_ml / df_test_ml for the rest of this notebook
df_train_ml = spark.read.parquet(TRAIN_ML_PATH_B).cache()
df_val_ml   = spark.read.parquet(VAL_ML_PATH_B).cache()
df_test_ml  = spark.read.parquet(TEST_ML_PATH_B).cache()

if VERIFY_ROW_COUNTS:
    n_train = df_train_ml.count()
    n_val   = df_val_ml.count()
    n_test  = df_test_ml.count()
    print(f"{'='*60}")
    print(f"  DATA SCALE (Notebook B / GBT split)")
    print(f"{'='*60}")
    print(f"  Train:      {n_train:>12,} rows")
    print(f"  Validation: {n_val:>12,} rows")
    print(f"  Test:       {n_test:>12,} rows")
    print(f"  Total:      {n_train + n_val + n_test:>12,} rows")
else:
    n_train = df_train_ml.count()  # one count for §3 diagnostics if needed
    print("VERIFY_ROW_COUNTS=False: skipping val/test counts on load.")
    print(f"  Train rows (single count): {n_train:,}")

_dim = df_train_ml.first()["features"].size
print(f"  Feature vector size: {_dim}")



VERIFY_ROW_COUNTS=False: skipping val/test counts on load.
  Train rows (single count): 14,515,569
  Feature vector size: 104


In [0]:
# Class balance per split (skipped when VERIFY_ROW_COUNTS=False to avoid extra full scans)
if VERIFY_ROW_COUNTS:
    for name, df in [("Train", df_train_ml), ("Validation", df_val_ml), ("Test", df_test_ml)]:
        total   = df.count()
        delayed = df.filter(F.col(TARGET) == 1).count()
        ontime  = total - delayed
        print(f"  {name:12s}: {total:>12,} | Delayed: {delayed:>10,} ({100*delayed/total:.1f}%) "
              f"| On-time: {ontime:>10,} ({100*ontime/total:.1f}%)")
else:
    print("VERIFY_ROW_COUNTS=False: skipping per-split class balance counts.")



VERIFY_ROW_COUNTS=False: skipping per-split class balance counts.


## 2. Evaluation Helpers — F-β Score (β=2)

All model comparisons use **F-β with β=2** as the primary metric.

$$F_\beta = \frac{(1+\beta^2) \cdot \text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

With β=2 this becomes:
$$F_2 = \frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

AUROC is retained as a secondary metric to assess rank-ordering ability across all thresholds.

In [0]:
def fbeta(precision, recall, beta=BETA):
    """Compute F-β score."""
    denom = beta**2 * precision + recall
    if denom < 1e-12:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def evaluate_model(model, df, label_col=TARGET, threshold=None):
    """
    Evaluate a fitted PySpark ML model.
    If threshold is not None, override the default 0.5 classification threshold.
    Returns a dict of metrics including F-β (primary), F1, AUROC, and confusion matrix.
    """
    predictions = model.transform(df)

    if threshold is not None:
        predictions = predictions.withColumn(
            "prediction",
            F.when(vector_to_array(F.col("probability"))[1] >= threshold, 1.0).otherwise(0.0),
        )

    auroc = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction", labelCol=label_col, metricName="areaUnderROC"
    ).evaluate(predictions)

    tp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 1)).count()
    fp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 0)).count()
    fn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 1)).count()
    tn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 0)).count()

    prec_del = tp / max(tp + fp, 1)
    rec_del  = tp / max(tp + fn, 1)
    f1_del   = 2 * prec_del * rec_del / max(prec_del + rec_del, 1e-12)
    fb_del   = fbeta(prec_del, rec_del)

    accuracy = (tp + tn) / max(tp + fp + fn + tn, 1)

    return {
        f"F{BETA:.0f} (delayed)": round(fb_del, 4),    # PRIMARY
        "F1 (delayed)": round(f1_del, 4),
        "Precision (delayed)": round(prec_del, 4),
        "Recall (delayed)": round(rec_del, 4),
        "AUROC": round(auroc, 4),
        "Accuracy": round(accuracy, 4),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


def print_metrics(metrics, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    for k, v in metrics.items():
        if k in ("TP", "FP", "FN", "TN"):
            print(f"  {k:25s}: {v:>12,}")
        else:
            marker = "  <-- PRIMARY" if f"F{BETA:.0f}" in k and "delayed" in k else ""
            print(f"  {k:25s}: {v:>12.4f}{marker}")

def time_series_cv(df, estimator_fn, n_folds=5, label_col=TARGET, window_frac=0.20):
    """
    Rolling-window time-series cross-validation on training data.

    Unlike expanding-window CV (where train grows each fold), rolling-window
    uses a FIXED-SIZE training window that slides forward in time. This means:
      - Each fold trains on the same amount of data
      - Earlier data is dropped as the window advances
      - Better reflects real-world deployment where only recent data is relevant
      - Prevents early folds from dominating due to larger training sizes

    window_frac : fraction of total rows used for each fold's training window (default 20%)
    Each fold's validation window is the next (1/n_folds * remaining) chunk after train.

    Fold layout example with n_folds=3, window_frac=0.20:
      Fold 1: Train [0%–20%]   → Val [20%–33%]
      Fold 2: Train [13%–33%]  → Val [33%–47%]
      Fold 3: Train [27%–47%]  → Val [47%–60%]
    """
    # Sort by fl_date_sort if available so fold boundaries are truly temporal.
    if "fl_date_sort" in df.columns:
        df_indexed = df.orderBy("fl_date_sort").withColumn("_row_id", F.monotonically_increasing_id()).cache()
    else:
        print("WARNING: 'fl_date_sort' not in DataFrame — fold boundaries may not be temporal. "
              "Re-run Phase 2.4 to add fl_date_sort to ML parquets.")
        df_indexed = df.withColumn("_row_id", F.monotonically_increasing_id()).cache()
    total = df_indexed.count()
    fold_metrics = []

    # Val window size: split remaining data (after first train window) into n_folds chunks
    val_frac = (1.0 - window_frac) / n_folds

    for fold in range(n_folds):
        val_start  = window_frac + fold * val_frac
        val_end    = val_start + val_frac
        train_start = val_start - window_frac  # fixed-size window ending at val_start

        quantiles = df_indexed.approxQuantile(
            "_row_id",
            [train_start, val_start, val_end],
            0.01
        )
        train_lo, train_hi, val_hi = quantiles

        fold_train = df_indexed.filter(
            (F.col("_row_id") >= train_lo) & (F.col("_row_id") < train_hi)
        ).drop("_row_id")

        fold_val = df_indexed.filter(
            (F.col("_row_id") >= train_hi) & (F.col("_row_id") < val_hi)
        ).drop("_row_id")

        n_ft, n_fv = fold_train.count(), fold_val.count()
        if n_fv == 0 or n_ft == 0:
            continue

        print(f"  Fold {fold+1}: Train {n_ft:,} (~{100*window_frac:.0f}% window) "
              f"| Val {n_fv:,}")
        model   = estimator_fn().fit(fold_train)
        metrics = evaluate_model(model, fold_val, label_col)
        metrics["fold"] = fold + 1
        fold_metrics.append(metrics)
        print(f"    F{BETA:.0f}(delayed)={metrics[f'F{BETA:.0f} (delayed)']:.4f}  "
              f"F1={metrics['F1 (delayed)']:.4f}  AUROC={metrics['AUROC']:.4f}")

    df_indexed.unpersist()
    return fold_metrics

print("Evaluation helpers defined.")

Evaluation helpers defined.


## 3. Class weights — not used for GBT

`GBTClassifier` does **not** support `weightCol`. Train rows match **Phase 3.3** `df_train_phase3_downsampled` (§4 aliases `df_train_ml`); no second undersample in this notebook. The `class_weight` column in the parquet is ignored here.



In [0]:
# No class-weight setup required for GBT.
pass



## 4. Train distribution (Phase 3.3 parquet)

Class balance is set **only** in Phase 3.3 (`UNDERSAMPLE_MAJORITY_RATIO` on `df_train_phase3_downsampled.parquet`). This section caches `df_train_ml` as `df_train_us_gbt` for downstream cells; **no** additional majority sampling in Phase 2.5. Val/test stay the natural mix.



In [0]:
n_delayed_b = df_train_ml.filter(F.col(TARGET) == 1).count()
n_ontime_b  = df_train_ml.filter(F.col(TARGET) == 0).count()

print("GBT train (Notebook B ML parquet = Phase 3.3 downsampled export):")
print(f"  On-time:  {n_ontime_b:>12,}")
print(f"  Delayed:  {n_delayed_b:>12,}")
print(f"  Ratio:    {n_ontime_b/max(n_delayed_b,1):.2f}:1")

df_train_us_gbt = df_train_ml.cache()
print(f"GBT train rows (no Phase 2.5 resample): {df_train_us_gbt.count():,}")



GBT train (Notebook B ML parquet = Phase 3.3 downsampled export):
  On-time:    10,886,398
  Delayed:     3,629,171
  Ratio:    3.00:1
GBT train rows (no Phase 2.5 resample): 14,515,569


## 5. Gradient Boosted Trees — Grid Search + Threshold Tuning

GBT achieves the strongest AUROC among the Phase 2 baselines (Colton's analysis).
`GBTClassifier` in PySpark does **not** support `weightCol`, so we rely on:
1. Train class balance from **Phase 3.3** parquet (single undersample)
2. Threshold tuning (prediction-time adjustment)
3. Grid search over `maxDepth` and `maxIter` (Phase 3 addition)

In [0]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol=TARGET,
    maxIter=50,
    maxDepth=5,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42,
)

print("Training GBT baseline (Phase 3.3 train distribution)...")
t0 = time.time()
gbt_model = gbt.fit(df_train_us_gbt)
gbt_time  = time.time() - t0
print(f"Done in {gbt_time:.1f}s")

gbt_val_default = evaluate_model(gbt_model, df_val_ml)
print_metrics(gbt_val_default, "GBT baseline — VALIDATION")

Training GBT baseline (Phase 3.3 train distribution)...
Done in 602.0s

  GBT baseline — VALIDATION
  F2 (delayed)             :       0.2251  <-- PRIMARY
  F1 (delayed)             :       0.2645
  Precision (delayed)      :       0.3735
  Recall (delayed)         :       0.2047
  AUROC                    :       0.6884
  Accuracy                 :       0.8180
  TP                       :      207,157
  FP                       :      347,502
  FN                       :      804,737
  TN                       :    4,970,142


### GBT Grid Search

Grid search over tree depth and iteration count. We use AUROC as the CV metric
(it evaluates the full ranking, not a threshold-dependent point), then select the best
model and apply threshold tuning for final F-β optimisation.

In [0]:
evaluator_auroc = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction", labelCol=TARGET, metricName="areaUnderROC"
)
THRESHOLDS = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]


gbt_gs = GBTClassifier(
    featuresCol="features",
    labelCol=TARGET,
    subsamplingRate=0.8,
    seed=42,
)
param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt_gs.maxDepth, [5, 7, 9])
    .addGrid(gbt_gs.maxIter,  [50, 100])
    .addGrid(gbt_gs.stepSize, [0.05, 0.10])
    .build()
)
cv_gbt = CrossValidator(
    estimator=gbt_gs,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_auroc,
    numFolds=3,
    parallelism=2,
    seed=42,
)

print("GBT grid search (12 combos × 3 folds = 36 fits) on Phase 3.3 train...")
t0 = time.time()
cv_gbt_model = cv_gbt.fit(df_train_us_gbt)
gbt_gs_time  = time.time() - t0
print(f"Done in {gbt_gs_time:.1f}s")

best_gbt = cv_gbt_model.bestModel
print(f"\nBest GBT: maxDepth={best_gbt._java_obj.getMaxDepth()}, "
      f"maxIter={best_gbt._java_obj.getMaxIter()}, "
      f"stepSize={best_gbt._java_obj.getStepSize():.2f}")

print("\nAll GBT grid search results:")
for params, score in zip(param_grid_gbt, cv_gbt_model.avgMetrics):
    d  = params[gbt_gs.maxDepth]
    it = params[gbt_gs.maxIter]
    lr = params[gbt_gs.stepSize]
    print(f"  maxDepth={d}, maxIter={it}, stepSize={lr} -> CV AUROC={score:.4f}")

best_gbt_val = evaluate_model(best_gbt, df_val_ml)
print_metrics(best_gbt_val, "Best GBT (grid search) — VALIDATION")

GBT grid search (12 combos × 3 folds = 36 fits) on Phase 3.3 train...
Done in 26882.2s

Best GBT: maxDepth=9, maxIter=100, stepSize=0.10

All GBT grid search results:
  maxDepth=5, maxIter=50, stepSize=0.05 -> CV AUROC=0.7023
  maxDepth=5, maxIter=50, stepSize=0.1 -> CV AUROC=0.7097
  maxDepth=5, maxIter=100, stepSize=0.05 -> CV AUROC=0.7096
  maxDepth=5, maxIter=100, stepSize=0.1 -> CV AUROC=0.7167
  maxDepth=7, maxIter=50, stepSize=0.05 -> CV AUROC=0.7117
  maxDepth=7, maxIter=50, stepSize=0.1 -> CV AUROC=0.7199
  maxDepth=7, maxIter=100, stepSize=0.05 -> CV AUROC=0.7203
  maxDepth=7, maxIter=100, stepSize=0.1 -> CV AUROC=0.7285
  maxDepth=9, maxIter=50, stepSize=0.05 -> CV AUROC=0.7211
  maxDepth=9, maxIter=50, stepSize=0.1 -> CV AUROC=0.7303
  maxDepth=9, maxIter=100, stepSize=0.05 -> CV AUROC=0.7301
  maxDepth=9, maxIter=100, stepSize=0.1 -> CV AUROC=0.7400

  Best GBT (grid search) — VALIDATION
  F2 (delayed)             :       0.3003  <-- PRIMARY
  F1 (delayed)             :   

### GBT Threshold Tuning

In [0]:
val_preds_gbt = best_gbt.transform(df_val_ml).cache()

print(f"GBT Threshold Sweep (validation set):")
print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

best_gbt_thresh, best_gbt_fb = 0.5, 0.0

for thresh in THRESHOLDS:
    preds = val_preds_gbt.withColumn(
        "pred_adj",
        F.when(vector_to_array(F.col("probability"))[1] >= thresh, 1.0).otherwise(0.0)
    )
    tp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 1)).count()
    fp = preds.filter((F.col("pred_adj") == 1) & (F.col(TARGET) == 0)).count()
    fn = preds.filter((F.col("pred_adj") == 0) & (F.col(TARGET) == 1)).count()
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    f1   = 2 * prec * rec / max(prec + rec, 1e-12)
    fb   = fbeta(prec, rec)
    marker = " <-- best" if fb > best_gbt_fb else ""
    if fb > best_gbt_fb:
        best_gbt_thresh, best_gbt_fb = thresh, fb
    print(f"  {thresh:>8.2f}  {prec:>10.4f}  {rec:>10.4f}  {f1:>10.4f}  {fb:>10.4f}{marker}")

print(f"\nBest GBT threshold: {best_gbt_thresh} (F{BETA:.0f}={best_gbt_fb:.4f})")
val_preds_gbt.unpersist()

best_gbt_val_tuned = evaluate_model(best_gbt, df_val_ml, threshold=best_gbt_thresh)
print_metrics(best_gbt_val_tuned, f"Best GBT (threshold={best_gbt_thresh}) — VALIDATION")

GBT Threshold Sweep (validation set):
    Thresh   Precision      Recall          F1          F2
  --------  ----------  ----------  ----------  ----------
      0.01      0.1599      1.0000      0.2757      0.4876 <-- best
      0.02      0.1599      1.0000      0.2757      0.4876
      0.03      0.1599      1.0000      0.2757      0.4876 <-- best
      0.05      0.1605      0.9994      0.2765      0.4886 <-- best
      0.07      0.1639      0.9937      0.2814      0.4938 <-- best
      0.10      0.1727      0.9723      0.2934      0.5049 <-- best
      0.15      0.1911      0.9093      0.3158      0.5190 <-- best
      0.20      0.2130      0.8209      0.3383      0.5226 <-- best
      0.25      0.2365      0.7219      0.3563      0.5118
      0.30      0.2599      0.6235      0.3668      0.4871
      0.35      0.2820      0.5309      0.3684      0.4513
      0.40      0.3028      0.4461      0.3607      0.4075
      0.45      0.3242      0.3668      0.3441      0.3574
      0.50    

### Why we chose this GBT threshold

GBT produces well-calibrated probability scores — its sigmoid transformation of the raw
margin gives a meaningful probability of delay. The default 0.5 threshold assumes a
symmetric cost: it is optimal only when a false positive and false negative are equally
costly. Since we use β=2 (recall costs more), the operational threshold should sit below
0.5 in almost all cases.

The swept threshold selected above is the point where:
- **Recall** is high enough to catch the majority of real delays
- **Precision** has not yet dropped so severely that F-β (β=2) starts declining
- Specifically, the score peaks at the threshold where the marginal recall gain from
  lowering the threshold further is outweighed by the precision loss

This threshold is locked from the validation sweep and applied once to the blind test set.

## 5b. XGBoost — Grid Search + Threshold Tuning

XGBoost typically outperforms PySpark GBT on tabular data. It supports native
class imbalance handling via `scale_pos_weight`, so we train on the **full**
train set (no undersampling needed).
`xgboost.spark` is available on Databricks Runtime 10.4 ML+.

In [0]:
try:
    from xgboost.spark import SparkXGBClassifier

    _n_delayed = df_train_ml.filter(F.col(TARGET) == 1).count()
    _n_ontime  = df_train_ml.filter(F.col(TARGET) == 0).count()
    _spw = _n_ontime / max(_n_delayed, 1)
    print(f"XGBoost scale_pos_weight = {_spw:.2f}  ({_n_ontime:,} ontime / {_n_delayed:,} delayed)")

    xgb = SparkXGBClassifier(
        features_col="features",
        label_col=TARGET,
        scale_pos_weight=_spw,
        max_depth=6,
        n_estimators=200,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="aucpr",
        use_gpu=False,
        seed=42,
    )

    print("Training XGBoost (scale_pos_weight, full train)...")
    t0 = time.time()
    xgb_model = xgb.fit(df_train_ml)
    xgb_time  = time.time() - t0
    print(f"Done in {xgb_time:.1f}s")

    xgb_val_default = evaluate_model(xgb_model, df_val_ml)
    print_metrics(xgb_val_default, "XGBoost (threshold=0.5) — VALIDATION")

    # XGBoost threshold sweep
    _xgb_val_pd = (
        xgb_model.transform(df_val_ml)
        .select(vector_to_array(F.col("probability"))[1].alias("prob"), F.col(TARGET).alias("label"))
        .toPandas()
    )
    _xgb_probs  = _xgb_val_pd["prob"].values
    _xgb_labels = _xgb_val_pd["label"].values

    print(f"\nXGBoost Threshold Sweep (validation set):")
    print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
    print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

    best_xgb_thresh, best_xgb_fb = 0.5, 0.0
    for thresh in THRESHOLDS:
        _pred = (_xgb_probs >= thresh).astype(int)
        _tp = ((_pred == 1) & (_xgb_labels == 1)).sum()
        _fp = ((_pred == 1) & (_xgb_labels == 0)).sum()
        _fn = ((_pred == 0) & (_xgb_labels == 1)).sum()
        _prec = _tp / max(_tp + _fp, 1)
        _rec  = _tp / max(_tp + _fn, 1)
        _f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
        _fb   = fbeta(_prec, _rec)
        marker = " <-- best" if _fb > best_xgb_fb else ""
        if _fb > best_xgb_fb:
            best_xgb_thresh, best_xgb_fb = thresh, _fb
        print(f"  {thresh:>8.2f}  {_prec:>10.4f}  {_rec:>10.4f}  {_f1:>10.4f}  {_fb:>10.4f}{marker}")

    print(f"\nBest XGBoost threshold: {best_xgb_thresh} (F{BETA:.0f}={best_xgb_fb:.4f})")

    xgb_val_tuned = evaluate_model(xgb_model, df_val_ml, threshold=best_xgb_thresh)
    print_metrics(xgb_val_tuned, f"XGBoost (threshold={best_xgb_thresh}) — VALIDATION")

    # ── GBT + XGBoost Ensemble ────────────────────────────────────────────────
    _gbt_val_pd = (
        best_gbt.transform(df_val_ml)
        .select(vector_to_array(F.col("probability"))[1].alias("p_gbt"), F.col(TARGET).alias("label"))
        .toPandas()
    )
    _ens_probs  = 0.5 * _gbt_val_pd["p_gbt"].values + 0.5 * _xgb_probs
    _ens_labels = _gbt_val_pd["label"].values

    print(f"\nGBT+XGBoost Ensemble Threshold Sweep (validation, equal weights):")
    print(f"  {'Thresh':>8s}  {'Precision':>10s}  {'Recall':>10s}  {'F1':>10s}  {'F{:.0f}'.format(BETA):>10s}")
    print(f"  {'-'*8}  {'-'*10}  {'-'*10}  {'-'*10}  {'-'*10}")

    best_gbt_xgb_thresh, best_gbt_xgb_fb = 0.5, 0.0
    for thresh in THRESHOLDS:
        _pred = (_ens_probs >= thresh).astype(int)
        _tp = ((_pred == 1) & (_ens_labels == 1)).sum()
        _fp = ((_pred == 1) & (_ens_labels == 0)).sum()
        _fn = ((_pred == 0) & (_ens_labels == 1)).sum()
        _prec = _tp / max(_tp + _fp, 1)
        _rec  = _tp / max(_tp + _fn, 1)
        _f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
        _fb   = fbeta(_prec, _rec)
        marker = " <-- best" if _fb > best_gbt_xgb_fb else ""
        if _fb > best_gbt_xgb_fb:
            best_gbt_xgb_thresh, best_gbt_xgb_fb = thresh, _fb
        print(f"  {thresh:>8.2f}  {_prec:>10.4f}  {_rec:>10.4f}  {_f1:>10.4f}  {_fb:>10.4f}{marker}")

    print(f"\nBest GBT+XGB ensemble threshold: {best_gbt_xgb_thresh} (F{BETA:.0f}={best_gbt_xgb_fb:.4f})")
    print(f"(GBT alone: F{BETA:.0f}={best_gbt_fb:.4f} | XGB alone: F{BETA:.0f}={best_xgb_fb:.4f})")

    # Ensemble blind test
    _gbt_test_pd = (
        best_gbt.transform(df_test_ml)
        .select(vector_to_array(F.col("probability"))[1].alias("p_gbt"), F.col(TARGET).alias("label"))
        .toPandas()
    )
    _xgb_test_pd = (
        xgb_model.transform(df_test_ml)
        .select(vector_to_array(F.col("probability"))[1].alias("p_xgb"))
        .toPandas()
    )
    _ens_test  = 0.5 * _gbt_test_pd["p_gbt"].values + 0.5 * _xgb_test_pd["p_xgb"].values
    _ens_tlabs = _gbt_test_pd["label"].values
    _pred_test = (_ens_test >= best_gbt_xgb_thresh).astype(int)
    _tp = ((_pred_test == 1) & (_ens_tlabs == 1)).sum()
    _fp = ((_pred_test == 1) & (_ens_tlabs == 0)).sum()
    _fn = ((_pred_test == 0) & (_ens_tlabs == 1)).sum()
    _tn = ((_pred_test == 0) & (_ens_tlabs == 0)).sum()
    _prec = _tp / max(_tp + _fp, 1)
    _rec  = _tp / max(_tp + _fn, 1)
    _f1   = 2 * _prec * _rec / max(_prec + _rec, 1e-12)
    _fb   = fbeta(_prec, _rec)
    print(f"\nGBT+XGB Ensemble (thresh={best_gbt_xgb_thresh}) — BLIND TEST")
    print(f"  F{BETA:.0f} (delayed):        {_fb:.4f}  <-- PRIMARY")
    print(f"  F1 (delayed):          {_f1:.4f}")
    print(f"  Precision (delayed):   {_prec:.4f}")
    print(f"  Recall (delayed):      {_rec:.4f}")
    print(f"  TP={_tp:,}  FP={_fp:,}  FN={_fn:,}  TN={_tn:,}")

    _XGB_AVAILABLE = True

except ImportError:
    print("xgboost.spark not available — skipping XGBoost and ensemble sections.")
    print("Install: pip install xgboost  (requires Databricks Runtime 10.4 ML+)")
    _XGB_AVAILABLE = False
    best_xgb_thresh, best_xgb_fb = None, None
    xgb_model = None

XGBoost scale_pos_weight = 3.00  (10,886,398 ontime / 3,629,171 delayed)
Training XGBoost (scale_pos_weight, full train)...


2026-04-18 21:25:15,332 INFO XGBoost-PySpark: _fit Running xgboost-3.0.0 on 1 workers with
	booster params: {'objective': 'binary:logistic', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'aucpr', 'learning_rate': 0.05, 'max_depth': 6, 'scale_pos_weight': 2.999692767301403, 'subsample': 0.8, 'seed': 42, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 200}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[21:30:04] [0]	training-aucpr:0.40768
[21:30:13] [1]	training-aucpr:0.41817
[21:30:23] [2]	training-aucpr:0.42116
[21:30:31] [3]	training-aucpr:0.42259
[21:30:41] [4]	training-aucpr:0.42478
[21:30:51] [5]	training-aucpr:0.42609
[21:31:01] [6]	training-aucpr:0.42688
[21:31:11] [7]	training-aucpr:0.42726
[21:31:21] [8]	training-aucpr:0.42882
[21:31:31] [9]	training-aucpr:0.42904
[21:31:41] [10]	training-aucpr:0.42945
[21:31:51] [11]	training-aucpr:0.43034
[21:32:01] [12]	training-aucpr:0.43103
[21:32:12] [13]	training-aucpr:0.43167
[21:32:22] [1

Done in 2261.1s

  XGBoost (threshold=0.5) — VALIDATION
  F2 (delayed)             :       0.5113  <-- PRIMARY
  F1 (delayed)             :       0.3528
  Precision (delayed)      :       0.2326
  Recall (delayed)         :       0.7299
  AUROC                    :       0.6932
  Accuracy                 :       0.5718
  TP                       :      738,606
  FP                       :    2,437,049
  FN                       :      273,288
  TN                       :    2,880,595

XGBoost Threshold Sweep (validation set):
    Thresh   Precision      Recall          F1          F2
  --------  ----------  ----------  ----------  ----------
      0.01      0.1599      1.0000      0.2757      0.4876 <-- best
      0.02      0.1599      1.0000      0.2757      0.4876
      0.03      0.1599      1.0000      0.2757      0.4876
      0.05      0.1599      1.0000      0.2757      0.4876
      0.07      0.1599      1.0000      0.2757      0.4876
      0.10      0.1600      0.9999      0.2758

## 6. Time-Series Cross-Validation

Rolling-window CV validates that model performance is stable across time periods.

In [0]:
fb_col   = f"F{BETA:.0f} (delayed)"
best_gbt_depth = best_gbt._java_obj.getMaxDepth()
best_gbt_iter  = best_gbt._java_obj.getMaxIter()
best_gbt_step  = best_gbt._java_obj.getStepSize()

print(f"=== GBT Time-Series CV (5 folds, rolling window, full fold train) ===")
print(f"    Using: maxDepth={best_gbt_depth}, maxIter={best_gbt_iter}, stepSize={best_gbt_step:.2f}")

def make_gbt_fold_fn(depth, n_iter, step):
    def _fn():
        class GBTFoldEstimator:
            def fit(self, df):
                return GBTClassifier(
                    featuresCol="features", labelCol=TARGET,
                    maxDepth=depth, maxIter=n_iter, stepSize=step,
                    subsamplingRate=0.8, seed=42,
                ).fit(df)
        return GBTFoldEstimator()
    return _fn

gbt_cv_results = time_series_cv(
    df_train_ml,
    make_gbt_fold_fn(best_gbt_depth, best_gbt_iter, best_gbt_step)
)
gbt_cv_df = pd.DataFrame(gbt_cv_results)
print(f"\nGBT CV: F{BETA:.0f}(delayed) = {gbt_cv_df[fb_col].mean():.4f} ± {gbt_cv_df[fb_col].std():.4f}")
print(f"        AUROC           = {gbt_cv_df['AUROC'].mean():.4f} ± {gbt_cv_df['AUROC'].std():.4f}")


=== GBT Time-Series CV (5 folds, rolling window, full fold train) ===
    Using: maxDepth=9, maxIter=100, stepSize=0.10
  Fold 1: Train 2,817,581 (~20% window) | Val 2,327,644
    F2(delayed)=0.2027  F1=0.2608  AUROC=0.6987
  Fold 2: Train 2,800,536 (~20% window) | Val 2,386,773
    F2(delayed)=0.1638  F1=0.2228  AUROC=0.6990
  Fold 3: Train 2,915,241 (~20% window) | Val 2,351,900
    F2(delayed)=0.2142  F1=0.2724  AUROC=0.6884
  Fold 4: Train 3,020,127 (~20% window) | Val 2,291,060
    F2(delayed)=0.2257  F1=0.2903  AUROC=0.6929
  Fold 5: Train 2,799,885 (~20% window) | Val 2,426,296
    F2(delayed)=0.2420  F1=0.3059  AUROC=0.7046

GBT CV: F2(delayed) = 0.2097 ± 0.0295
        AUROC           = 0.6967 ± 0.0062


## 7. Blind Test Evaluation

Final evaluation on the held-out test set. Thresholds selected on validation are applied
here. The test set is examined **once** — these are the reported Phase 3 results.

In [0]:
print("Evaluating on blind test set (GBT only)...\n")

test_best_gbt = evaluate_model(best_gbt, df_test_ml, threshold=best_gbt_thresh)
print_metrics(test_best_gbt, f"Best GBT (thresh={best_gbt_thresh})         — BLIND TEST")



Evaluating on blind test set (GBT only)...


  Best GBT (thresh=0.2)         — BLIND TEST
  F2 (delayed)             :       0.5267  <-- PRIMARY
  F1 (delayed)             :       0.3452
  Precision (delayed)      :       0.2193
  Recall (delayed)         :       0.8110
  AUROC                    :       0.6805
  Accuracy                 :       0.4811
  TP                       :      853,574
  FP                       :    3,039,334
  FN                       :      198,902
  TN                       :    2,149,198


## 8. Summary Table

In [0]:
all_results = []

model_configs = [
    (f"GBT (grid search, thresh={best_gbt_thresh})", best_gbt, df_val_ml, df_test_ml, best_gbt_thresh, gbt_gs_time),
]

for model_name, model, val_df, test_df, thresh, t_time in model_configs:
    v = evaluate_model(model, val_df, threshold=thresh)
    t = evaluate_model(model, test_df, threshold=thresh)
    for split_name, metrics in [("Validation", v), ("Test (Blind)", t)]:
        row = {"Model": model_name, "Split": split_name, **metrics,
               "Train Time (s)": round(t_time, 1) if split_name == "Validation" else ""}
        all_results.append(row)

summary_df = pd.DataFrame(all_results)
print("\n" + "=" * 90)
print("  GBT SPLIT — RESULTS SUMMARY (primary metric: F-β, β=2)")
print("=" * 90)
display(spark.createDataFrame(summary_df))




  GBT SPLIT — RESULTS SUMMARY (primary metric: F-β, β=2)


/databricks/spark/python/pyspark/sql/pandas/conversion.py:473: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert '' with type str: tried to convert to double
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Model,Split,F2 (delayed),F1 (delayed),Precision (delayed),Recall (delayed),AUROC,Accuracy,TP,FP,FN,TN,Train Time (s)
"GBT (grid search, thresh=0.2)",Validation,0.5226,0.3383,0.213,0.8209,0.6954,0.4865,830658,3068888,181236,2248756,26882.2
"GBT (grid search, thresh=0.2)",Test (Blind),0.5267,0.3452,0.2193,0.811,0.6805,0.4811,853574,3039334,198902,2149198,


In [0]:
# Save results summary (GBT split — separate file from LR/RF split)
summary_df.to_csv(f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/phase3_results_summary_gbt.csv", index=False)
print(f"Results saved to {BASE_GROUP}/phase3_results_summary_gbt.csv")

Results saved to dbfs:/student-groups/Group_01_02/phase3_results_summary_gbt.csv


In [0]:
# Save trained models to DBFS for cross-notebook ensemble
WRITE_MODELS = True  # Set False to skip (saves ~2–5 min on large clusters)

if WRITE_MODELS:
    _model_base = f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/phase3_models"
    best_gbt.write().overwrite().save(f"{_model_base}/best_gbt_us")
    print(f"Saved: {_model_base}/best_gbt_us")
    if _XGB_AVAILABLE and xgb_model is not None:
        xgb_model.write().overwrite().save(f"{_model_base}/xgb_model_us")
        print(f"Saved: {_model_base}/xgb_model_us")
    print("Model saving complete. Load in phase2.5_ensemble.py for cross-notebook ensemble.")



Saved: /dbfs/student-groups/Group_01_02/phase3_models/best_gbt_us
Saved: /dbfs/student-groups/Group_01_02/phase3_models/xgb_model_us
Model saving complete. Load in phase2.5_ensemble.py for cross-notebook ensemble.


## 9. Feature Importance (Top / bottom 25 + optional CSV)

Named importances use `phase3_final_num_cols.json` order. Bottom 25 supports validation ablations; full CSV is written for offline notes.

In [0]:
importances = best_gbt.featureImportances.toArray()
top_25_idx  = sorted(range(len(importances)), key=lambda i: importances[i], reverse=True)[:25]

print("Top 25 Features (Best GBT):")
print(f"  {'Rank':>4s}  {'Index':>6s}  {'Importance':>12s}")
print(f"  {'-'*4}  {'-'*6}  {'-'*12}")
for rank, idx in enumerate(top_25_idx, 1):
    print(f"  {rank:>4d}  [{idx:>4d}]  {importances[idx]:>12.6f}")

print("\nNote: indices correspond to the feature vector assembled by Phase 2.4.")
print("Feature names are in phase3_final_num_cols.json under BASE_GROUP.")

Top 25 Features (Best GBT):
  Rank   Index    Importance
  ----  ------  ------------
     1  [   2]      0.076341
     2  [   5]      0.063905
     3  [  37]      0.062625
     4  [   3]      0.061674
     5  [  40]      0.037243
     6  [  11]      0.030771
     7  [  15]      0.025094
     8  [  38]      0.021268
     9  [  95]      0.021214
    10  [  32]      0.020365
    11  [  48]      0.020018
    12  [  16]      0.019645
    13  [  35]      0.018707
    14  [  59]      0.017513
    15  [  31]      0.017394
    16  [  52]      0.017302
    17  [  17]      0.016772
    18  [   1]      0.016376
    19  [  18]      0.016242
    20  [  36]      0.016130
    21  [  68]      0.015600
    22  [  54]      0.015248
    23  [  63]      0.013593
    24  [  47]      0.013473
    25  [  50]      0.013195

Note: indices correspond to the feature vector assembled by Phase 2.4.
Feature names are in phase3_final_num_cols.json under BASE_GROUP.


In [0]:
# Attempt to load feature names from Phase 3.3 JSON and display named importance
try:
    import json
    raw = dbutils.fs.head(f"{BASE_GROUP}/phase3_final_num_cols.json", max_bytes=1_000_000)
    feature_names = json.loads(raw)
    named_imp = [
        (feature_names[i] if i < len(feature_names) else f"feat_{i}", float(importances[i]))
        for i in top_25_idx
    ]
    imp_df = pd.DataFrame(named_imp, columns=["feature", "importance"])
    imp_df.index = range(1, len(imp_df)+1)
    print("Top 25 Named Feature Importances (GBT):")
    display(spark.createDataFrame(imp_df))
except Exception as e:
    print(f"Could not load feature names: {e}")
    print("Run Phase 2.4 / Phase 3.3 first to generate phase3_final_num_cols.json.")

Top 25 Named Feature Importances (GBT):


feature,importance
DAY_OF_MONTH,0.07634145439677346
CRS_DEP_TIME,0.06390451129581833
carrier_delay_rate,0.06262516087529252
YEAR,0.06167387673054416
route_delay_rate,0.0372432992799493
dep_hour_sin,0.030771023479895524
day_sin,0.025093641594871666
origin_delay_rate,0.02126831689218655
temp_anomaly_station_month,0.021213692483650713
dest_train_departures,0.020364648469424446


In [0]:
# Bottom 25 by split gain + full importance CSV (same JSON order as Phase 2.4 assembler)
try:
    import json
    raw = dbutils.fs.head(f"{BASE_GROUP}/phase3_final_num_cols.json", max_bytes=1_000_000)
    _fnames = json.loads(raw)
except Exception:
    _fnames = []

_imp = best_gbt.featureImportances.toArray()
_n = len(_imp)
_bottom = sorted(range(_n), key=lambda i: _imp[i])[:25]

print("Bottom 25 Features (Best GBT, lowest importances):")
print(f"  {'Rank':>4s}  {'Index':>6s}  {'Importance':>12s}  Feature")
print(f"  {'-'*4}  {'-'*6}  {'-'*12}  {'-'*48}")
for rank, idx in enumerate(_bottom, 1):
    name = _fnames[idx] if idx < len(_fnames) else f"feat_{idx}"
    print(f"  {rank:>4d}  [{idx:>4d}]  {_imp[idx]:>12.6f}  {name}")

_all = pd.DataFrame(
    [
        (_fnames[i] if i < len(_fnames) else f"feat_{i}", float(_imp[i]))
        for i in range(_n)
    ],
    columns=["feature", "importance"],
).sort_values("importance", ascending=False)

_csv_path = f"{BASE_GROUP.replace('dbfs:', '/dbfs')}/feature_importance_gbt.csv"
_all.to_csv(_csv_path, index=False)
print(f"\nFull table: {BASE_GROUP}/feature_importance_gbt.csv ({_n} rows)")

Bottom 25 Features (Best GBT, lowest importances):
  Rank   Index    Importance  Feature
  ----  ------  ------------  ------------------------------------------------
     1  [  10]      0.000008  is_late_night
     2  [  34]      0.000010  is_hub_dest
     3  [   8]      0.000010  is_evening_peak
     4  [  29]      0.000015  wx_peak_x_strong_wind
     5  [   7]      0.000017  is_morning_peak
     6  [  28]      0.000049  wx_evening_x_bad_vis
     7  [  90]      0.000119  peak_wind_x_vis_bin
     8  [  66]      0.000123  busy_origin_x_peak_wx
     9  [ 100]      0.000200  late_night_x_wind_scl
    10  [  27]      0.000205  is_origin_busy_2h_before
    11  [  89]      0.000228  morning_x_wx_rh_precip
    12  [  72]      0.000233  wx_morning_peak_x_vis_scl
    13  [ 101]      0.000238  late_night_x_vis_bin
    14  [  33]      0.000274  is_hub_origin
    15  [  24]      0.000330  is_holiday_weekend_adjacent
    16  [  99]      0.000368  late_night_x_precip_scl
    17  [  67]      0.0004

## 10. Key Findings (GBT split)

### Train distribution (Phase 3.3 parquet)

GBT trains on the **Phase 3.3** downsampled train parquet (Notebook **B** ML export; assembler-only — non-`*_scl` in raw parquet units, no `StandardScaler`). **No** second undersample in this notebook. Validation and test remain the **natural** class mix.

### F-β (β=2) and threshold tuning

Thresholds are chosen on the **validation** set to maximize **F-β**; that cutoff is frozen for the **blind test**.

### Feature importances

Indices align with `phase3_final_num_cols.json` (Phase 3.3 manifest). For the full **LR + RF + GBT** narrative and a single combined table, run [phase2.5_modeling_final.ipynb](phase2.5_modeling_final.ipynb) on a large enough cluster or merge CSVs from both split notebooks.

### Next steps

See the monolithic Phase 2.5 notebook Key Findings for extended bullets (XGBoost, graph features, error analysis).

**Validation ablations:** remove **one** feature family per experiment in Phase 3.3, rebuild 3.3 → 2.4 A/B → 2.5, and compare **validation** AUROC / PR-AUC / F2 only. Run a **blind test once** for the final configuration.

